In [13]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import csv
import time
import logging

class LocalChScraper:
    def __init__(self, base_url="https://www.local.ch/fr/s/%20vétérinaire?rid=983969"):
        self.base_url = base_url
        self.driver = None
        self.results = []
        
        # Setup logging (file + console, safe for notebooks)
        self.logger = logging.getLogger("LocalChScraper")
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            log_filename = f"scraping_log_{time.strftime('%Y%m%d_%H%M%S')}.log"
            file_handler = logging.FileHandler(log_filename, encoding='utf-8')
            console_handler = logging.StreamHandler()
            formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
            file_handler.setFormatter(formatter)
            console_handler.setFormatter(formatter)
            self.logger.addHandler(file_handler)
            self.logger.addHandler(console_handler)

    def setup_driver(self):
        """Initialize the Chrome WebDriver with appropriate options."""
        options = webdriver.ChromeOptions()
        # Removed --headless to see what's happening and avoid anti-bot detection
        options.add_argument('--disable-gpu')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36')
        options.add_argument('--window-size=1920,1080')
        
        self.driver = webdriver.Chrome(options=options)
        self.driver.implicitly_wait(10)

    def get_page_data(self, page_number=1):
        """Scrape data from a specific page."""
        url = f"{self.base_url}&page={page_number}" if page_number > 1 else self.base_url
        
        try:
            self.logger.info(f"Loading page: {url}")
            self.driver.get(url)
            
            # Wait for page to load
            time.sleep(8)  # Increased wait time
            
            # Print page title for debugging
            self.logger.info(f"Page title: {self.driver.title}")
            
            # Debug: Print page source sample to see what's actually loaded
            page_source = self.driver.page_source
            self.logger.info(f"Page source length: {len(page_source)}")
            self.logger.info(f"Page source sample: {page_source[:500]}...")
            
            # Try multiple possible selectors for cards
            card_selectors = [
                "article[data-testid^='list-element-desktop']",  # Original selector
                "article[data-testid*='list-element']",
                "article[class*='card']",
                "div[class*='result']",
                "div[class*='listing']",
                "[data-testid*='result']",
                "article",
                ".result-item",
                "[class*='business']",
                "[class*='clinic']",
                "div[class*='item']",
                "div[class*='entry']"
            ]
            
            cards = []
            for selector in card_selectors:
                try:
                    cards = self.driver.find_elements(By.CSS_SELECTOR, selector)
                    self.logger.info(f"Selector '{selector}': Found {len(cards)} elements")
                    if cards and len(cards) > 0:
                        self.logger.info(f"Using selector: {selector}")
                        # Show first card content for debugging
                        if len(cards) > 0:
                            first_card_text = cards[0].text.strip()
                            self.logger.info(f"First card content: {first_card_text[:200]}...")
                        break
                except Exception as e:
                    self.logger.debug(f"Selector '{selector}' failed: {str(e)}")
                    continue
            
            if not cards:
                self.logger.warning("No cards found with any selector")
                # Try to find ANY elements that might contain the data
                all_elements = self.driver.find_elements(By.CSS_SELECTOR, "*")
                self.logger.info(f"Total elements on page: {len(all_elements)}")
                
                # Look for elements containing "vétérinaire" or "clinic"
                vet_elements = self.driver.find_elements(By.XPATH, "//*[contains(text(), 'vétérinaire') or contains(text(), 'clinic') or contains(text(), 'tierarzt')]")
                self.logger.info(f"Elements containing 'vétérinaire': {len(vet_elements)}")
                
                return []
            
            page_results = []
            for i, card in enumerate(cards):
                try:
                    # Skip if card doesn't contain expected content
                    card_text = card.text.strip()
                    if not card_text or len(card_text) < 10:
                        continue
                    
                    result = {
                        'title': '',
                        'address': '',
                        'link': ''
                    }
                    
                    # Extract title - try multiple selectors
                    title_selectors = [
                        "h2[data-testid='title']",  # Original selector
                        "h1", "h2", "h3", "h4",
                        "[class*='title']",
                        "[class*='name']",
                        "a[href*='/d/']"
                    ]
                    
                    for selector in title_selectors:
                        try:
                            title_elem = card.find_element(By.CSS_SELECTOR, selector)
                            if title_elem.text.strip():
                                result['title'] = title_elem.text.strip()
                                break
                        except:
                            continue
                    
                    # Extract address
                    try:
                        address_elem = card.find_element(By.TAG_NAME, "address")
                        result['address'] = address_elem.text.strip()
                    except:
                        # Try other address selectors
                        address_selectors = [
                            "[class*='address']",
                            "[class*='location']",
                            "[itemprop='address']"
                        ]
                        for selector in address_selectors:
                            try:
                                addr_elem = card.find_element(By.CSS_SELECTOR, selector)
                                if addr_elem.text.strip():
                                    result['address'] = addr_elem.text.strip()
                                    break
                            except:
                                continue
                    
                    # Extract link
                    try:
                        link_elem = card.find_element(By.XPATH, "./parent::a")
                        result['link'] = link_elem.get_attribute('href')
                    except:
                        try:
                            link_elem = card.find_element(By.CSS_SELECTOR, "a[href*='/d/']")
                            result['link'] = link_elem.get_attribute('href')
                        except:
                            try:
                                link_elem = card.find_element(By.TAG_NAME, "a")
                                href = link_elem.get_attribute('href')
                                if href and '/d/' in href:
                                    result['link'] = href
                            except:
                                pass
                    
                    # Only add if we have at least title and link
                    if result['title'] and result['link']:
                        page_results.append(result)
                        self.logger.info(f"Extracted: {result['title']}")
                    else:
                        self.logger.warning(f"Skipping incomplete card {i}: {card_text[:100]}...")
                    
                except Exception as e:
                    self.logger.error(f"Error extracting card {i}: {str(e)}")
                    continue
                    
            return page_results
            
        except TimeoutException:
            self.logger.error(f"Timeout while loading page {page_number}")
            return []
        except Exception as e:
            self.logger.error(f"Error processing page {page_number}: {str(e)}")
            return []

    def save_to_csv(self, filename='veterinary_clinics.csv'):
        """Save the scraped results to a CSV file."""
        if not self.results:
            self.logger.warning("No results to save")
            return
            
        try:
            with open(filename, 'w', newline='', encoding='utf-8-sig') as f:
                writer = csv.DictWriter(f, fieldnames=['title', 'address', 'link'])
                writer.writeheader()
                writer.writerows(self.results)
            self.logger.info(f"Results saved to {filename}")
        except Exception as e:
            self.logger.error(f"Error saving results to CSV: {str(e)}")

    def scrape(self, max_pages=None):
        """Main scraping function."""
        try:
            self.setup_driver()
            page_number = 1
            
            while True:
                self.logger.info(f"Scraping page {page_number}")
                page_results = self.get_page_data(page_number)
                
                if not page_results:
                    self.logger.info("No more results found")
                    break
                    
                self.results.extend(page_results)
                
                if max_pages and page_number >= max_pages:
                    self.logger.info(f"Reached maximum number of pages ({max_pages})")
                    break
                    
                page_number += 1
                time.sleep(2)  # Be nice to the server
                
        except Exception as e:
            self.logger.error(f"Error during scraping: {str(e)}")
        finally:
            if self.driver:
                self.driver.quit()

def debug_single_page():
    """Debug function to test just one page and see what's happening."""
    scraper = LocalChScraper()
    try:
        scraper.setup_driver()
        scraper.get_page_data(1)  # Just test page 1
    except Exception as e:
        scraper.logger.error(f"Debug error: {str(e)}")
    finally:
        if scraper.driver:
            scraper.driver.quit()

def main():
    scraper = LocalChScraper()
    scraper.scrape(max_pages=58)  # Test with first 3 pages
    scraper.save_to_csv()

if __name__ == "__main__":
    # Run the full scraper now (saves CSV and logs to file)
    main()
    # If needed later for troubleshooting, call: debug_single_page()

2025-10-15 23:02:17,307 - INFO - Scraping page 1
2025-10-15 23:02:17,307 - INFO - Scraping page 1
2025-10-15 23:02:17,316 - INFO - Loading page: https://www.local.ch/fr/s/%20vétérinaire?rid=983969
2025-10-15 23:02:17,316 - INFO - Loading page: https://www.local.ch/fr/s/%20vétérinaire?rid=983969
2025-10-15 23:02:35,715 - INFO - Page title: vétérinaire – Trouvez un large choix sur local.ch
2025-10-15 23:02:35,715 - INFO - Page title: vétérinaire – Trouvez un large choix sur local.ch
2025-10-15 23:02:35,907 - INFO - Page source length: 1062885
2025-10-15 23:02:35,907 - INFO - Page source length: 1062885
2025-10-15 23:02:35,909 - INFO - Page source sample: <html lang="fr" class="notranslate" translate="no"><head><script async="" src="https://www.googletagmanager.com/gtm.js?id=GTM-NT77MB"></script><link rel="stylesheet" href="/_next/static/css/3d64279488d93dc0.css" data-precedence="next"><link rel="stylesheet" href="/_next/static/css/b90424fadd8cca1b.css" data-precedence="next"><link rel="s

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException
import pandas as pd
import csv
import time
import logging
import re
import os
from datetime import datetime
from functools import wraps

def retry_on_exception(retries=3, delay=5):
    """Retry decorator with exponential backoff."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            retry_count = 0
            while retry_count < retries:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    retry_count += 1
                    if retry_count == retries:
                        raise e
                    wait_time = delay * (2 ** (retry_count - 1))  # Exponential backoff
                    logging.warning(f"Attempt {retry_count} failed. Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
            return None
        return wrapper
    return decorator

class VetDetailScraper:
    def __init__(self):
        self.driver = None
        self.results = []
        self.checkpoint_file = 'scraping_checkpoint.csv'
        self.processed_urls = set()
        
        # Setup logging
        log_filename = f'scraping_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler(log_filename),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)

    def setup_driver(self):
        """Initialize the Chrome WebDriver with appropriate options."""
        options = webdriver.ChromeOptions()
        options.add_argument('--headless')
        options.add_argument('--disable-gpu')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        
        self.driver = webdriver.Chrome(options=options)
        self.driver.implicitly_wait(10)

    def load_checkpoint(self):
        """Load previously processed URLs from checkpoint file."""
        if os.path.exists(self.checkpoint_file):
            try:
                df = pd.read_csv(self.checkpoint_file, encoding='utf-8-sig')
                self.processed_urls = set(df['url'].tolist())
                self.results = df.to_dict('records')
                self.logger.info(f"Loaded {len(self.processed_urls)} processed URLs from checkpoint")
            except Exception as e:
                self.logger.error(f"Error loading checkpoint: {str(e)}")
                self.processed_urls = set()
                self.results = []

    def save_checkpoint(self):
        """Save current progress to checkpoint file."""
        try:
            if self.results:
                df = pd.DataFrame(self.results)
                df.to_csv(self.checkpoint_file, index=False, encoding='utf-8-sig')
                self.logger.info(f"Saved checkpoint with {len(self.results)} records")
        except Exception as e:
            self.logger.error(f"Error saving checkpoint: {str(e)}")

    def clean_time_text(self, text):
        """Clean and format time text."""
        if not text or text.lower() == 'fermé':
            return 'Fermé'
            
        text = re.sub(r'\s+', ' ', text)
        text = text.replace('jusqu\'à', '-')
        text = text.replace(' / ', ', ')
        text = re.sub(r'(\d):(\d{2})', r'\1:\2', text)
        text = re.sub(r'\s*-\s*', '-', text)
        text = re.sub(r'\s*,\s*', ', ', text)
        
        return text.strip()

    @retry_on_exception(retries=3, delay=5)
    def parse_opening_hours(self, element):
        """Parse opening hours for each day."""
        days = {
            'Lundi': 'Fermé', 
            'Mardi': 'Fermé', 
            'Mercredi': 'Fermé', 
            'Jeudi': 'Fermé',
            'Vendredi': 'Fermé', 
            'Samedi': 'Fermé', 
            'Dimanche': 'Fermé'
        }
        
        time_frames = element.find_elements(
            By.CSS_SELECTOR, 
            "li[data-cy='opening-hours-weekdays']"
        )
        
        for frame in time_frames:
            day = frame.find_element(
                By.CLASS_NAME, 
                "TimeFrame_day__3_oHv"
            ).text
            
            hours_element = frame.find_element(
                By.CSS_SELECTOR, 
                "div[itemprop='openingHours']"
            )
            hours = hours_element.text.strip()
            
            if day in days:
                days[day] = self.clean_time_text(hours)
                
        return days

    def clean_description(self, text):
        """Clean and format description text."""
        if not text:
            return ''
            
        text = re.sub(r'\n+', '\n', text)
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'<[^>]+>', '', text)
        
        return text.strip()

    def clean_list_text(self, text):
        """Clean and format list text."""
        if not text:
            return ''
            
        text = re.sub(r',\s*,', ',', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip(',').strip()

    def parse_address(self, address_text):
        """Parse address into components."""
        if not address_text:
            return '', '', '', ''
            
        address_text = address_text.replace('&nbsp;', ' ')
        address_text = re.sub(r'\s+', ' ', address_text.strip())
        
        pattern = r'(.*?)\s*,?\s*(\d{4})\s*(.*?)(?:\s*\((.*?)\))?$'
        match = re.match(pattern, address_text)
        
        if match:
            street = match.group(1).strip()
            zipcode = match.group(2)
            city = match.group(3).strip()
            kanton = match.group(4) or ''
            return street, zipcode, city, kanton.strip()
            
        return address_text, '', '', ''

    @retry_on_exception(retries=3, delay=5)
    def scrape_detail_page(self, url):
        """Scrape data from a detail page."""
        self.driver.get(url)
        
        WebDriverWait(self.driver, 10).until(
            EC.presence_of_element_located(
                (By.CLASS_NAME, "detail_detail__SXBfi")
            )
        )
        
        detail_data = {
            'url': url,
            'logo_url': '',
            'title': '',
            'street': '',
            'zipcode': '',
            'city': '',
            'kanton': '',
            'phone_numbers': '',
            'email': '',
            'website': '',
            'description': '',
            'languages': ''
        }
        
        # Get logo URL
        try:
            logo_img = self.driver.find_element(
                By.CSS_SELECTOR,
                ".DetailHeaderRow_logoContainer__kdz6N img"
            )
            detail_data['logo_url'] = logo_img.get_attribute('src')
        except NoSuchElementException:
            pass
        
        # Get title
        try:
            title = self.driver.find_element(
                By.CSS_SELECTOR,
                "[data-cy='header-title']"
            )
            detail_data['title'] = title.text.strip()
        except NoSuchElementException:
            pass
        
        # Get address
        try:
            address = self.driver.find_element(
                By.CLASS_NAME,
                "DetailMapPreview_addressValue__pQROv"
            )
            street, zipcode, city, kanton = self.parse_address(address.text)
            detail_data.update({
                'street': street,
                'zipcode': zipcode,
                'city': city,
                'kanton': kanton
            })
        except NoSuchElementException:
            pass
        
        # Get opening hours
        try:
            hours_element = self.driver.find_element(
                By.CLASS_NAME,
                "OpeningHours_openingHoursBody__ApmXd"
            )
            opening_hours = self.parse_opening_hours(hours_element)
            detail_data.update(opening_hours)
        except NoSuchElementException:
            pass
        
        # Get contact information
        for contact_group in self.driver.find_elements(
            By.CLASS_NAME,
            "ContactGroupsAccordion_contactGroup__dsb2_"
        ):
            try:
                label = contact_group.find_element(
                    By.CLASS_NAME,
                    "ContactGroupsAccordion_contactType__8Y1ED"
                ).text.lower()
                
                value = contact_group.find_element(
                    By.CLASS_NAME,
                    "ContactGroupsAccordion_accordionGroupValue__lmVyw"
                ).text.strip()
                
                if 'téléphone' in label:
                    detail_data['phone_numbers'] = self.clean_list_text(value)
                elif 'e-mail' in label:
                    detail_data['email'] = value
                elif 'site web' in label:
                    detail_data['website'] = value
            except NoSuchElementException:
                continue
        
        # Get languages
        try:
            headers = self.driver.find_elements(
                By.CLASS_NAME,
                "DescriptionContent_detailListTitle__hIIB6"
            )
            
            languages_section = None
            for header in headers:
                if header.text.strip() == "Langues":
                    languages_section = header
                    break
            
            if languages_section:
                languages_dd = languages_section.find_element(
                    By.XPATH,
                    "./following-sibling::dd[1]"
                )
                
                language_spans = languages_dd.find_elements(
                    By.CLASS_NAME,
                    "DescriptionContent_detailListContentAttribute__zhs_H"
                )
                
                langs = [span.text.strip().rstrip(',') for span in language_spans]
                detail_data['languages'] = self.clean_list_text(', '.join(langs))
        except NoSuchElementException:
            pass
        
        return detail_data

    def scrape_from_links(self, csv_file):
        """Main function to scrape details from links in CSV."""
        try:
            # Load existing progress
            self.load_checkpoint()
            
            # Read all links
            df = pd.read_csv(csv_file)
            links = df['link'].tolist()
            total_links = len(links)
            
            self.setup_driver()
            
            for i, link in enumerate(links, 1):
                if link in self.processed_urls:
                    self.logger.info(f"Skipping already processed link {i}/{total_links}: {link}")
                    continue
                    
                self.logger.info(f"Scraping detail page {i}/{total_links}: {link}")
                
                try:
                    detail_data = self.scrape_detail_page(link)
                    
                    if detail_data:
                        self.results.append(detail_data)
                        self.processed_urls.add(link)
                        
                        # Save checkpoint every 5 records
                        if len(self.results) % 5 == 0:
                            self.save_checkpoint()
                    
                except Exception as e:
                    self.logger.error(f"Error processing link {link}: {str(e)}")
                    # Save checkpoint on error
                    self.save_checkpoint()
                
                time.sleep(2)
            
            # Final save
            self.save_checkpoint()
            
        except Exception as e:
            self.logger.error(f"Error during scraping: {str(e)}")
        finally:
            if self.driver:
                self.driver.quit()

def main():
    scraper = VetDetailScraper()
    scraper.scrape_from_links('veterinary_clinics.csv')

if __name__ == "__main__":
    main()

2025-10-17 20:46:10,070 - ERROR - Error during scraping: [Errno 2] No such file or directory: 'veterinary_clinics.csv'
